In [1]:
# install all dependencies
from pathlib import Path
import pandas as pd
import krippendorff
from irrCAC.raw import CAC
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

In [3]:
#Gwet's AC2 calculation using irrCAC package

#irrCAC uses data matrix with rows as items and columns as raters
df = pd.read_csv(DATA_DIR/"Final_LongMetrics_NoAccuracy.csv")
metrics = ['Clarity', 'Grade_Score', 'Appropriateness', 'Jargon_Avoided']

def get_gwet_ac2(df, metric):
    matrix = df.pivot(index='Article_ID', columns='Participant_IDs', values=metric)
    
    print(f"Gwet's AC2 for {metric}:")
    cac_abstractors=CAC(matrix, weights='ordinal')
    output = cac_abstractors.gwet()['est']

    output_summary = pd.DataFrame([{
    'Metric': output['coefficient_name'],
    'Value': f"{output['coefficient_value']:.3f}",
    'Std Error': f"{output['se']:.4f}",
    '95% CI': f"[{output['confidence_interval'][0]:.3f}, {output['confidence_interval'][1]:.3f}]",
    'z-stat': f"{output['z']:.4f}",
    'p-value': f"{output['p_value']:.4e}",
    'Observed Agreement (Pa)': f"{output['pa']:.4f}",
    'Expected Agreement (Pe)': f"{output['pe']:.4f}"
    }])

    print(output_summary.to_string(index=False), "\n")

for metric in metrics:
    get_gwet_ac2(df, metric)
    


Gwet's AC2 for Clarity:
Metric Value Std Error         95% CI  z-stat    p-value Observed Agreement (Pa) Expected Agreement (Pe)
   AC2 0.540    0.0494 [0.442, 0.638] 10.9338 0.0000e+00                  0.8395                  0.6512 

Gwet's AC2 for Grade_Score:
Metric Value Std Error         95% CI  z-stat    p-value Observed Agreement (Pa) Expected Agreement (Pe)
   AC2 0.693    0.0351 [0.624, 0.763] 19.7532 0.0000e+00                  0.8959                  0.6605 

Gwet's AC2 for Appropriateness:
Metric Value Std Error         95% CI  z-stat    p-value Observed Agreement (Pa) Expected Agreement (Pe)
   AC2 0.552    0.0496 [0.454, 0.651] 11.1424 0.0000e+00                  0.8375                  0.6370 

Gwet's AC2 for Jargon_Avoided:
Metric Value Std Error         95% CI  z-stat    p-value Observed Agreement (Pa) Expected Agreement (Pe)
   AC2 0.556    0.0494 [0.458, 0.654] 11.2614 0.0000e+00                  0.8363                  0.6314 



In [4]:
# 1. Load your data
df = pd.read_csv(DATA_DIR/"Final_LongMetrics_NoAccuracy.csv")

def get_alpha_and_ci(df, column_name, n_bootstraps=1000, random_state=42):
    """
    Calculates Krippendorff's Alpha and its 95% Confidence Interval via bootstrapping.
    """
    # 1. Pivot to the Rater x Paper matrix
    matrix = df.pivot(index='Participant_IDs', columns='Article_ID', values=column_name)
    matrix_values = matrix.values
    
    # 2. Calculate the point estimate
    alpha_point = krippendorff.alpha(reliability_data=matrix_values, level_of_measurement='ordinal')
    
    # 3. Bootstrapping procedure
    np.random.seed(random_state)
    n_raters, n_items = matrix_values.shape
    bootstrapped_alphas = []
    
    for _ in range(n_bootstraps):
        # Resample items (columns) with replacement
        resampled_indices = np.random.choice(n_items, size=n_items, replace=True)
        resampled_matrix = matrix_values[:, resampled_indices]
        
        try:
            # Calculate alpha for the simulated matrix
            alpha_boot = krippendorff.alpha(reliability_data=resampled_matrix, level_of_measurement='ordinal')
            
            # Filter out invalid calculations (e.g., matrices where chance agreement = 1.0 due to resampling identical scores)
            if not np.isnan(alpha_boot):
                bootstrapped_alphas.append(alpha_boot)
        except Exception:
            # Bypasses iteration if the resampled matrix lacks sufficient variance
            continue
            
    # 4. Extract 95% CI bounds from the empirical distribution
    if bootstrapped_alphas:
        lower_bound = np.percentile(bootstrapped_alphas, 2.5)
        upper_bound = np.percentile(bootstrapped_alphas, 97.5)
    else:
        lower_bound, upper_bound = np.nan, np.nan
        
    return alpha_point, lower_bound, upper_bound

# Execution across domains
metrics = ['Clarity', 'Grade_Score','Appropriateness', 'Jargon_Avoided']

print("Krippendorff's Alpha Estimates with 95% CIs:\n" + "-"*35)
for metric in metrics:
    alpha, lower, upper = get_alpha_and_ci(df, metric, n_bootstraps=1000)
    print(f"{metric}: α = {alpha:.3f}, 95% CI [{lower:.3f}, {upper:.3f}]")

Krippendorff's Alpha Estimates with 95% CIs:
-----------------------------------
Clarity: α = 0.141, 95% CI [0.017, 0.266]
Grade_Score: α = 0.071, 95% CI [-0.054, 0.183]
Appropriateness: α = 0.066, 95% CI [-0.052, 0.179]
Jargon_Avoided: α = 0.093, 95% CI [-0.026, 0.203]


In [3]:
# Group by Text_ID and see the spread of scores
text_stats = df.groupby('Article_ID')['Clarity'].agg(['mean', 'std', 'count', 'min', 'max'])

# Look for texts where the 'std' (Standard Deviation) is high
# e.g., a text where one person gave a 2 and another gave a 9
troublesome_texts = text_stats[text_stats['std'] > 1].sort_values(by='std', ascending=False)

print("Texts with the most disagreement:")
print(troublesome_texts)

Texts with the most disagreement:
                mean       std  count  min  max
Article_ID                                     
95          2.500000  2.121320      2    1    4
48          3.500000  2.121320      2    2    5
34          2.666667  2.081666      3    1    5
90          3.000000  2.000000      3    1    5
11          4.000000  1.732051      3    2    5
65          4.000000  1.732051      3    2    5
4           3.000000  1.732051      3    2    5
87          3.333333  1.527525      3    2    5
73          3.666667  1.527525      3    2    5
72          3.666667  1.527525      3    2    5
27          3.666667  1.527525      3    2    5
29          3.666667  1.527525      3    2    5
31          3.333333  1.527525      3    2    5
33          3.666667  1.527525      3    2    5
60          3.333333  1.527525      3    2    5
59          3.666667  1.527525      3    2    5
2           3.666667  1.527525      3    2    5
18          3.666667  1.527525      3    2    5
49    

In [55]:
df

,Participant_IDs,Group,Article_ID,Experience_Level,Clarity,Grade_Score,Appropriateness,Jargon_Avoided
0,P-001,1,51,3,4,3,5,5
1,P-001,1,52,3,4,3,3,4
2,P-001,1,53,3,5,3,5,5
3,P-001,1,54,3,5,3,5,5
4,P-001,1,55,3,4,3,4,5
...,...,...,...,...,...,...,...,...
245,P-025,10,41,4,4,1,4,3
246,P-025,10,42,4,5,1,5,5
247,P-025,10,43,4,5,1,5,5
248,P-025,10,44,4,4,1,5,5


In [ ]:
import statsmodels.formula.api as smf

df = pd.read_csv(DATA_DIR/"Final_LongMetrics_NoAccuracy.csv")

def get_icc_with_ci(df, metric, n_bootstraps=1000):
    # original fit
    model = smf.mixedlm(
        f"{metric} ~ 1", #Fixed Effects with mean intercept, no additional fixed predictors
        df,
        groups="Article_ID", #Random factor is article ID, for random intercept
        re_formula="1",
        vc_formula={"Participant_IDs": "0 + C(Participant_IDs)"} #nested factor for random intercepts of raters
    )
    mdf = model.fit(disp=False)

    var_article = float(mdf.cov_re.iloc[0, 0])
    var_rater = float(mdf.vcomp[0]) if hasattr(mdf, "vcomp") and len(mdf.vcomp) > 0 else 0.0
    var_resid = float(mdf.scale)

    icc_point = var_article / (var_article + var_rater + var_resid) #equation for ICC 2,1 - article variance / total variance
    icc_point_consistency = var_article / (var_article + var_resid) #equation for ICC 3,1 - article variance / (article variance + residual variance)

    icc_boot = []
    icc_boot_consistency = []  # List to store bootstrapped ICC consistency values
    article_ids = df["Article_ID"].unique()

    for _ in range(n_bootstraps):
        boot_article_ids = np.random.choice(article_ids, size=len(article_ids), replace=True)

        boot_df_list = []
        for new_id, a_id in enumerate(boot_article_ids):
            paper_data = df[df["Article_ID"] == a_id].copy()
            paper_data["Boot_Article_ID"] = new_id
            boot_df_list.append(paper_data)

        boot_df = pd.concat(boot_df_list, ignore_index=True)

        try:
            boot_model = smf.mixedlm(
                f"{metric} ~ 1",
                boot_df,
                groups=boot_df["Boot_Article_ID"],
                re_formula="1",
                vc_formula={"Participant_IDs": "0 + C(Participant_IDs)"}
            )
            boot_mdf = boot_model.fit(disp=False)

            b_var_article = float(boot_mdf.cov_re.iloc[0, 0])
            b_var_rater = float(boot_mdf.vcomp[0]) if hasattr(boot_mdf, "vcomp") and len(boot_mdf.vcomp) > 0 else 0.0
            b_var_resid = float(boot_mdf.scale)

            boot_icc = b_var_article / (b_var_article + b_var_rater + b_var_resid)
            boot_icc_consistency = b_var_article / (b_var_article + b_var_resid) #equation for ICC 3,1 - article variance / (article variance + residual variance)
            icc_boot.append(boot_icc)
            icc_boot_consistency.append(boot_icc_consistency)
        except Exception:
            continue

    if len(icc_boot) > 0:
        lower_abs = np.percentile(icc_boot, 2.5)
        upper_abs = np.percentile(icc_boot, 97.5)
        lower_consistency = np.percentile(icc_boot_consistency, 2.5)
        upper_consistency = np.percentile(icc_boot_consistency, 97.5)
    else:
        lower_abs, upper_abs = np.nan, np.nan
        lower_consistency, upper_consistency = np.nan, np.nan

    return icc_point, lower_abs, upper_abs, icc_point_consistency, lower_consistency, upper_consistency

In [ ]:
# ICC with bootstrapped confidence intervals
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
# Suppress convergence warnings that happen during bootstrapping
warnings.simplefilter('ignore', ConvergenceWarning)
warnings.filterwarnings("ignore", message="Random effects covariance is singular")

metrics = ['Clarity', 'Grade_Score', 'Appropriateness', 'Jargon_Avoided']

for metric in metrics:
    icc_abs, low, high, icc_consistency, low_consistency, high_consistency = get_icc_with_ci(df, metric, n_bootstraps=1000)
    print(f"{metric}: ICC Absolute = {icc_abs:.3f} ({low:.3f}, {high:.3f})")
    print(f"{metric}: ICC Consistency = {icc_consistency:.3f} ({low_consistency:.3f}, {high_consistency:.3f})")

Clarity: ICC Absolute = 0.130 (0.011, 0.248)
Clarity: ICC Consistency = 0.264 (0.029, 0.415)
Grade_Score: ICC Absolute = 0.079 (0.000, 0.196)
Grade_Score: ICC Consistency = 0.178 (0.000, 0.355)
Appropriateness: ICC Absolute = 0.065 (0.000, 0.167)
Appropriateness: ICC Consistency = 0.153 (0.000, 0.317)
Jargon_Avoided: ICC Absolute = 0.047 (0.000, 0.162)
Jargon_Avoided: ICC Consistency = 0.114 (0.000, 0.310)


In [6]:
# ICC variance components + variations

import statsmodels.formula.api as smf

metrics = ['Clarity', 'Grade_Score', 'Appropriateness', 'Jargon_Avoided']

def get_icc_abs_and_consistency(df, metric):
    # 1. Fit LMM with Crossed Random Effects (Article_ID and Participant_IDs)
    # Using ~ 1 for unadjusted baseline ICC (add fixed effects like ~ Experience_Level if desired)
    model = smf.mixedlm(
        f"{metric} ~ 1", 
        df, 
        groups="Article_ID", 
        re_formula="1", 
        vc_formula={"Participant_IDs": "0 + C(Participant_IDs)"}
    )
    mdf = model.fit()

    # 2. Extract Variance Components
    var_article = float(mdf.cov_re.iloc[0, 0])  # Variance between Articles/Summaries
    var_rater = float(mdf.vcomp[0])              # Variance between Reviewers (Leniency/Harshness)
    var_resid = float(mdf.scale)                # Remaining Residual Noise

    # 3. Calculate ICC Variations
    icc_absolute = var_article / (var_article + var_rater + var_resid)
    icc_consistency = var_article / (var_article + var_resid)

    print(f"=== Metric: {metric} ===")
    print(f"Var(Article):     {var_article:.4f}")
    print(f"Var(Rater):       {var_rater:.4f}")
    print(f"Var(Residual):    {var_resid:.4f}")
    print(f"ICC (Absolute):   {icc_absolute:.3f}")
    print(f"ICC (Consistency): {icc_consistency:.3f}\n")

for metric in metrics:
    get_icc_abs_and_consistency(df, metric)

=== Metric: Clarity ===
Var(Article):     0.1504
Var(Rater):       0.5889
Var(Residual):    0.4199
ICC (Absolute):   0.130
ICC (Consistency): 0.264

=== Metric: Grade_Score ===
Var(Article):     0.1234
Var(Rater):       0.8751
Var(Residual):    0.5703
ICC (Absolute):   0.079
ICC (Consistency): 0.178

=== Metric: Appropriateness ===
Var(Article):     0.0696
Var(Rater):       0.6072
Var(Residual):    0.3855
ICC (Absolute):   0.065
ICC (Consistency): 0.153

=== Metric: Jargon_Avoided ===
Var(Article):     0.0496
Var(Rater):       0.6317
Var(Residual):    0.3848
ICC (Absolute):   0.047
ICC (Consistency): 0.114



In [77]:
#Full ICC metrics for baseline ICC (no fixed effects)
for metric in metrics:
    model = smf.mixedlm(
        f"{metric} ~ 1", 
        df, 
        groups="Article_ID", 
        re_formula="1", 
        vc_formula={"Participant_IDs": "0 + C(Participant_IDs)"}
    )
    mdf = model.fit()

    # Print the fixed effect table to see coefficient, p-value, and confidence intervals
    print(mdf.summary())

    # Specifically extract the p-value for Experience Level
    print("p-value:", mdf.pvalues)

            Mixed Linear Model Regression Results
Model:              MixedLM   Dependent Variable:   Clarity  
No. Observations:   250       Method:               REML     
No. Groups:         100       Scale:                0.4110   
Min. group size:    1         Log-Likelihood:       -369.9500
Max. group size:    4         Converged:            Yes      
Mean group size:    2.5                                      
-------------------------------------------------------------
                    Coef. Std.Err.   z    P>|z| [0.025 0.975]
-------------------------------------------------------------
Intercept           3.901    0.075 51.741 0.000  3.753  4.049
Article_ID Var      0.167                                    
Participant_IDs Var 0.561                                    

p-value: Intercept              0.0
Article_ID Var         NaN
Participant_IDs Var    NaN
dtype: float64
            Mixed Linear Model Regression Results
Model:              MixedLM  Dependent Variable:  

In [ ]:
#Check if experience level is a predictor of the scores given by the reviewers. This will help us understand if more experienced reviewers are more lenient or harsh in their scoring.
for metric in metrics:
    model = smf.mixedlm(
        f"{metric} ~ Experience_Level", 
        df, 
        groups="Article_ID", 
        re_formula="1", 
        vc_formula={"Participant_IDs": "0 + C(Participant_IDs)"}
    )
    mdf = model.fit()

    # Print the fixed effect table to see coefficient, p-value, and confidence intervals
    print(mdf.summary())

    # Specifically extract the p-value for Experience Level
    print("p-value:", mdf.pvalues)

           Mixed Linear Model Regression Results
Model:               MixedLM  Dependent Variable:  Clarity  
No. Observations:    250      Method:              REML     
No. Groups:          100      Scale:               0.3982   
Min. group size:     1        Log-Likelihood:      -367.2948
Max. group size:     4        Converged:           Yes      
Mean group size:     2.5                                    
------------------------------------------------------------
                    Coef. Std.Err.   z   P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept           3.018    0.306 9.857 0.000  2.418  3.618
Experience_Level    0.246    0.083 2.965 0.003  0.083  0.408
Article_ID Var      0.123                                   
Participant_IDs Var 0.576                                   

p-value: Intercept              6.363989e-23
Experience_Level       3.027232e-03
Article_ID Var                  NaN
Participant_IDs Var             NaN
dtyp

In [ ]:
# Pipeline for Accuracy Dataset Inter-rater reliability
df_acc = pd.read_csv(DATA_DIR/"Final_LongMetrics_OnlyAccuracy.csv")

metric = 'Accuracy'

#combine two scores (abstract and full text) columns
df_acc['Score'] = df_acc['Abstract_Score'].fillna(df_acc['Full_Text_Score'])

#remove any rows where Score is NaN (i.e., both Abstract_Score and Full_Text_Score are NaN)
df_acc = df_acc.dropna(subset=['Score'])

#rename column Accuracy_ID to Article_ID to match the other dataset
df_acc = df_acc.rename(columns={'Accuracy_ID': 'Article_ID'})

alpha, lower, upper = get_alpha_and_ci(df_acc, 'Score', n_bootstraps=1000)
print(f"{metric}: α = {alpha:.3f}, 95% CI [{lower:.3f}, {upper:.3f}]")

get_gwet_ac2(df_acc, 'Score')

icc_abs, low_abs, high_abs, icc_consistency, low_consistency, high_consistency = get_icc_with_ci(df_acc, 'Score', n_bootstraps=1000)
print(f"{metric}: ICC Absolute = {icc_abs:.3f}, 95% CI [{low_abs:.3f}, {high_abs:.3f}]")
print(f"{metric}: ICC Consistency = {icc_consistency:.3f}, 95% CI [{low_consistency:.3f}, {high_consistency:.3f}]")

get_icc_abs_and_consistency(df_acc, 'Score')

Accuracy: α = 0.080, 95% CI [-0.012, 0.178]
Gwet's AC2 for Score:
Metric  Value Std Error           95% CI  z-stat    p-value Observed Agreement (Pa) Expected Agreement (Pe)
   AC2 0.6180    0.0471 [0.5233, 0.7126] 13.1259 0.0000e+00                  0.8589                  0.6308 

Accuracy: ICC Absolute = 0.081, 95% CI [0.000, 0.197]
Accuracy: ICC Consistency = 0.183, 95% CI [0.000, 0.357]
=== Metric: Score ===
Var(Article):     0.0774
Var(Rater):       0.5321
Var(Residual):    0.3464
ICC (Absolute):   0.081
ICC (Consistency): 0.183



In [ ]:
#Alternative Krippendorff's Alpha, with Z-score normalization within a group of reviewers. (Within each group reviewers rate the same 10 articles, while between groups different articles)
# 1. Load your data
df = pd.read_csv(DATA_DIR/"Final_LongMetrics_NoAccuracy.csv")

def add_within_rater_zscore(df, column_name):
    """
    Calculates Z-score within each Participant_IDs.
    If a rater has 0 variance (gave the exact same score to every article), 
    sets their Z-score to 0.0 to avoid NaN division errors.
    """
    grouped = df.groupby('Participant_IDs')[column_name]
    means = grouped.transform('mean')
    stds = grouped.transform('std', ddof=1)
    
    z_scores = np.where(stds > 0, (df[column_name] - means) / stds, 0.0)
    # Preserve any original NaNs if a rater skipped an article
    z_scores = np.where(df[column_name].isna(), np.nan, z_scores)
    return z_scores


def get_alpha_and_ci(df, column_name, normalize=False, n_bootstraps=1000, random_state=42):
    """
    Calculates Pooled Krippendorff's Alpha and its 95% CI via Group-Stratified Bootstrapping,
    plus the per-group alphas across the 10 groups.
    """
    work_df = df.copy()
    
    # 1. Determine whether to use raw ordinal scores or Z-scored interval scores
    if normalize:
        target_col = f"{column_name}_Z"
        work_df[target_col] = add_within_rater_zscore(work_df, column_name)
        measurement_level = 'interval'  # Continuous Z-scores require 'interval'
    else:
        target_col = column_name
        measurement_level = 'ordinal'
        
    # 2. Pivot each of the 10 groups into its own Rater x Article block
    groups = sorted(work_df['Group'].unique())
    group_blocks = {}
    group_alphas = {}
    
    for g in groups:
        g_df = work_df[work_df['Group'] == g]
        block_matrix = g_df.pivot(index='Participant_IDs', columns='Article_ID', values=target_col).values.astype(float)
        group_blocks[g] = block_matrix
        
        # Calculate point estimate for this specific group
        try:
            group_alphas[g] = krippendorff.alpha(reliability_data=block_matrix, level_of_measurement=measurement_level)
        except Exception:
            group_alphas[g] = np.nan

    # 3. Calculate the Global Pooled Point Estimate (across the full 10-group matrix)
    full_matrix = work_df.pivot(index='Participant_IDs', columns='Article_ID', values=target_col).values.astype(float)
    alpha_point = krippendorff.alpha(reliability_data=full_matrix, level_of_measurement=measurement_level)
    
    # 4. Stratified Bootstrapping (resample articles WITHIN each of the 10 groups)
    np.random.seed(random_state)
    bootstrapped_alphas = []
    
    # Pre-calculate dimensions to build the block-diagonal matrix quickly during bootstrapping
    total_raters = sum(b.shape[0] for b in group_blocks.values())
    total_articles = sum(b.shape[1] for b in group_blocks.values())
    
    for _ in range(n_bootstraps):
        # Create an empty NaN matrix for the resampled full dataset
        resampled_full = np.full((total_raters, total_articles), np.nan)
        
        r_start, c_start = 0, 0
        for g in groups:
            block = group_blocks[g]
            n_r, n_c = block.shape
            
            # Resample columns (articles) with replacement within Group g
            resampled_cols = np.random.choice(n_c, size=n_c, replace=True)
            resampled_full[r_start:r_start+n_r, c_start:c_start+n_c] = block[:, resampled_cols]
            
            r_start += n_r
            c_start += n_c
            
        try:
            alpha_boot = krippendorff.alpha(reliability_data=resampled_full, level_of_measurement=measurement_level)
            if not np.isnan(alpha_boot):
                bootstrapped_alphas.append(alpha_boot)
        except Exception:
            continue
            
    # 5. Extract 95% CI bounds
    if bootstrapped_alphas:
        lower_bound = np.percentile(bootstrapped_alphas, 2.5)
        upper_bound = np.percentile(bootstrapped_alphas, 97.5)
    else:
        lower_bound, upper_bound = np.nan, np.nan
        
    return alpha_point, lower_bound, upper_bound, group_alphas


# Execution across domains
metrics = ['Clarity', 'Grade_Score', 'Appropriateness', 'Jargon_Avoided']

print("Krippendorff's Alpha: Raw (Ordinal) vs. Z-Scored (Interval)")
print("=" * 75)
print(f"{'Metric':<18} | {'Raw α [95% CI]':<24} | {'Z-Scored α [95% CI]':<24}")
print("-" * 75)

all_group_results = {}

for metric in metrics:
    # Run Raw (Ordinal)
    raw_a, raw_l, raw_u, raw_g = get_alpha_and_ci(df, metric, normalize=False, n_bootstraps=1000)
    
    # Run Z-Scored (Interval)
    z_a, z_l, z_u, z_g = get_alpha_and_ci(df, metric, normalize=True, n_bootstraps=1000)
    
    print(f"{metric:<18} | {raw_a:>6.3f} [{raw_l:>6.3f}, {raw_u:>6.3f}] | {z_a:>6.3f} [{z_l:>6.3f}, {z_u:>6.3f}]")
    
    # Store per-group Z-scored alphas so you can inspect Groups 1-10
    all_group_results[f"{metric}_Raw"] = raw_g
    all_group_results[f"{metric}_Z"] = z_g

# Print the breakdown by Group (1 to 10) to spot any low-performing groups
print("\nPer-Group Breakdown (Raw vs. Z-Scored α for each Group 1-10):")
print("-" * 75)
group_summary_df = pd.DataFrame(all_group_results)
group_summary_df.index.name = 'Group'
print(group_summary_df.round(3))

Krippendorff's Alpha: Raw (Ordinal) vs. Z-Scored (Interval)
Metric             | Raw α [95% CI]           | Z-Scored α [95% CI]     
---------------------------------------------------------------------------
Clarity            |  0.136 [-0.007,  0.280] |  0.184 [ 0.042,  0.305]
Grade_Score        | -0.021 [-0.162,  0.114] |  0.126 [-0.033,  0.256]
Appropriateness    |  0.078 [-0.058,  0.210] |  0.197 [ 0.056,  0.317]
Jargon_Avoided     |  0.081 [-0.054,  0.216] |  0.113 [-0.038,  0.248]

Per-Group Breakdown (Raw vs. Z-Scored α for each Group 1-10):
---------------------------------------------------------------------------
       Clarity_Raw  Clarity_Z  Grade_Score_Raw  Grade_Score_Z  \
Group                                                           
1            0.241      0.290            0.009          0.262   
2           -0.133      0.022           -0.258          0.066   
3           -0.120      0.329           -0.092          0.402   
4            0.224      0.247           -0.